# WEB DEV — Flask, HTML, CSS (9569 P2)

Lookup notebook for **S4B HTML**, **S4C CSS**, **S4D Flask**. This is a **Paper 2** skill set: files on disk, local server, browser.

## Folder contract (how detached HTML/CSS are handled)

Flask will not read HTML out of a notebook cell. Exam layout is always:

```text
task_folder/
  app.py                 (or this notebook's Flask cell)
  templates/
    home.html
    results.html
    add.html
  static/
    style.css
  STUDENT.csv / LATE.csv
  database.db            (created at run)
```

**This notebook is the source of truth.** Run the writer cells to emit `templates/` and `static/`. Edit the string in the notebook, re-run that cell, refresh the browser.

On the exam machine: paste the same HTML/CSS into Notepad++ as those filenames. Same bytes, different editor.

## Contents
1. [File writer](#writer)
2. [CSS](#css)
3. [HTML templates](#html)
4. [Flask app](#flask) — GET/POST, redirect, variable route, SQLite, INSERT
5. [P2 cheat-sheet](#cheatsheet)

Run cells **top to bottom**. Last cell starts the server (`http://127.0.0.1:5000`). Stop it with the interrupt button before re-running.


<a id="writer"></a>
## 1. File writer

One helper so HTML/CSS stay in the notebook for lookup, and on disk for Flask.


In [1]:
from pathlib import Path

# run this notebook from this folder (boilerplates)
ROOT = Path.cwd()

def write_web(relative_path, content):
    """Write HTML/CSS to disk so Flask can find it. Same layout as a P2 task folder."""
    path = ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")
    print("wrote", path.relative_to(ROOT))


<a id="css"></a>
## 2. CSS

Link from every page with `url_for('static', filename='style.css')`. Keep selectors simple: `.container`, `table`, `label`, `input[type="submit"]`.

Exam traps: submit button needs a visible colour contrast; `th`/`td` need `border-collapse`.


In [4]:
CSS = """
.container {
    width: 800px;
    margin: 50px auto;
    padding: 20px;
    border: 2px solid black;
    background-color: white;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
}

table, th, td {
    border: 1px solid orange;
    border-collapse: collapse;
    padding: 8px;
}

label, input, select {
    display: block;
    box-sizing: border-box;
    margin-bottom: 8px;
}

input[type="submit"] {
    background-color: black;
    color: white;
    padding: 10px 16px;
    border: none;
    cursor: pointer;
}

.error {
    color: red;
}
"""
write_web("static/style.css", CSS)


wrote static\style.css


<a id="html"></a>
## 3. HTML templates

Skeleton every page:

```html
<!doctype html>
<html>
<head>
<title>...</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<div class="container">
  ...
</div>
</body>
</html>
```

Jinja used here: `{{ var }}`, `{% if %}`, `{% for row in results %}`.  
Form: `method="POST"`, `name=` keys become `request.form['...']`.  
`action="/"` or `action="{{ url_for('add_late') }}"`.


### home.html — search form (POST → redirect)


In [5]:
HOME = """
<!doctype html>
<html>
<head>
<title>latecoming search</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<div class="container">
<h1>latecoming search</h1>
{% if error %}
<p class="error">{{ error }}</p>
{% endif %}
<!-- POST to home; request.form keys match name= on inputs -->
<form method="POST" action="/">
<label>reason</label>
<select name="category">
<option value="Heavy traffic">Heavy traffic</option>
<option value="Missed the bus">Missed the bus</option>
<option value="Overslept">Overslept</option>
<option value="Alarm clock did not ring">Alarm clock did not ring</option>
</select>
<label>student name</label>
<input type="text" name="name">
<input type="submit" value="search">
</form>
</div>
</body>
</html>
"""
write_web("templates/home.html", HOME)


wrote templates\home.html


### results.html — table of query rows


In [6]:
RESULTS = """
<!doctype html>
<html>
<head>
<title>results</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<div class="container">
<h1>results</h1>
<p>search results for <b>{{ name }}</b> / <b>{{ category }}</b></p>
{% if results %}
<table>
<thead>
<tr>
<th>student name</th>
<th>class group</th>
<th>date</th>
<th>reason</th>
</tr>
</thead>
<tbody>
{% for row in results %}
<tr>
<td>{{ row['Student_Name'] }}</td>
<td>{{ row['Class_Group'] }}</td>
<td>{{ row['Date'] }}</td>
<td>{{ row['Reason'] }}</td>
</tr>
{% endfor %}
</tbody>
</table>
{% else %}
<p>no latecoming records match the query</p>
{% endif %}
<form method="GET" action="/">
<input type="submit" value="return to homepage">
</form>
</div>
</body>
</html>
"""
write_web("templates/results.html", RESULTS)


wrote templates\results.html


### add.html — INSERT form (common second P2 page)


In [7]:
INSERT = """
<!doctype html>
<html>
<head>
<title>add late record</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<div class="container">
<h1>add late record</h1>
{% if error %}
<p class="error">{{ error }}</p>
{% endif %}
<form method="POST" action="{{ url_for('add_late') }}">
<label>student id</label>
<input type="text" name="stu_id">
<label>date (YYYY-MM-DD)</label>
<input type="text" name="date">
<label>reason</label>
<input type="text" name="reason">
<input type="submit" value="add">
</form>
<a href="{{ url_for('home') }}">back</a>
</div>
</body>
</html>
"""
write_web("templates/add.html", INSERT)


wrote templates\add.html


<a id="flask"></a>
## 4. Flask app

Pattern for every route:

```text
@app.route(path, methods=["GET", "POST"])
def name(...):
    if request.method == "POST":
        val = request.form.get("field", "").strip()
        # validate → INSERT or redirect(url_for(...))
    return render_template("page.html", ...)
```

- **GET** `/` shows the form
- **POST** `/` reads the form, `redirect` to a **variable route** `/results/<category>/<name>`
- results opens its **own** DB connection, JOINs, `fetchall`, closes
- `/add` INSERT then redirect home

`row_factory = sqlite3.Row` lets templates use `row['Student_Name']`. Without it, use `row[0]`.

Open `http://127.0.0.1:5000` after this cell. Do not use ports below 1024.


In [8]:
from flask import Flask, render_template, request, redirect, url_for
import sqlite3, csv

app = Flask(__name__)

def get_db():
    conn = sqlite3.connect("database.db")
    conn.row_factory = sqlite3.Row
    return conn

def seed_db():
    conn = get_db()
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS late")
    cur.execute("DROP TABLE IF EXISTS student")
    with open("STUDENT.csv") as file:
        reader = csv.reader(file)
        headers = next(reader)
        cur.execute(f"""
            CREATE TABLE student (
                {headers[0]} INTEGER PRIMARY KEY,
                {headers[1]} TEXT,
                {headers[2]} TEXT,
                {headers[3]} INTEGER
            )
        """)
        for row in reader:
            cur.execute(f"INSERT OR IGNORE INTO student ({', '.join(headers)}) VALUES (?,?,?,?)", row)
    with open("LATE.csv") as file:
        reader = csv.reader(file)
        headers = next(reader)
        cur.execute(f"""
            CREATE TABLE late (
                {headers[0]} TEXT,
                {headers[1]} INTEGER,
                {headers[2]} TEXT,
                FOREIGN KEY({headers[1]}) REFERENCES student({headers[1]})
            )
        """)
        for row in reader:
            cur.execute(f"INSERT OR IGNORE INTO late ({', '.join(headers)}) VALUES (?,?,?)", row)
    conn.commit()
    conn.close()

seed_db()

@app.route("/", methods=["GET", "POST"])
def home():
    error = None
    if request.method == "POST":
        category = request.form.get("category", "").strip()
        name = request.form.get("name", "").strip()
        if name and category:
            return redirect(url_for("results", category=category, name=name))
        error = "Please fill in all fields"
    return render_template("home.html", error=error)

@app.route("/results/<category>/<name>")
def results(category, name):
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT student.Student_Name, student.Class_Group, late.Date, late.Reason
        FROM student INNER JOIN late ON student.Student_ID = late.Student_ID
        WHERE student.Student_Name LIKE ? AND late.Reason = ?
    """, (f"%{name}%", category))
    data = cur.fetchall()
    conn.close()
    return render_template("results.html", results=data, category=category, name=name)

@app.route("/add", methods=["GET", "POST"])
def add_late():
    error = None
    if request.method == "POST":
        stu_id = request.form.get("stu_id", "").strip()
        date = request.form.get("date", "").strip()
        reason = request.form.get("reason", "").strip()
        if not (stu_id and date and reason):
            error = "Please fill in all fields"
        else:
            conn = get_db()
            cur = conn.cursor()
            cur.execute("INSERT INTO late VALUES (?,?,?)", (date, stu_id, reason))
            conn.commit()
            conn.close()
            return redirect(url_for("home"))
    return render_template("add.html", error=error)

# exam habit: only start the server in the main program
if __name__ == "__main__":
    app.run(port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [19/Aug/2026 11:01:01] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 11:01:01] "GET /static/style.css HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 11:01:01] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [19/Aug/2026 11:01:22] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 11:01:22] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [19/Aug/2026 11:02:06] "POST / HTTP/1.1" 302 -
127.0.0.1 - - [19/Aug/2026 11:02:06] "GET /results/Overslept/amy%20lim HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 11:02:06] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [19/Aug/2026 11:02:08] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 11:02:08] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [19/Aug/2026 11:02:12] "POST / HTTP/1.1" 302 -
127.0.0.1 - - [19/Aug/2026 11:02:12] "GET /results/Heavy%20traffic/amy%20lim HTTP/1.1" 200 -
127.0.0.1 - - [19/Aug/2026 11:02:12] "GET /static/style.css HTTP/1.1" 304 -


<a id="cheatsheet"></a>
## 5. P2 cheat-sheet

### Build order (timed paper)
1. Create `templates/` and `static/`
2. Write CSS + HTML (empty table / form first — prove the page loads)
3. Flask: `app = Flask(__name__)`, `@app.route("/")` + `render_template`
4. Add POST + `request.form`
5. Wire SQLite (`?` placeholders)
6. `redirect(url_for(...))` for the results/insert flow
7. `app.run()` — then open the URL in Chrome

### Must-remember
```text
from flask import Flask, render_template, request, redirect, url_for
app = Flask(__name__)
@app.route("/", methods=["GET", "POST"])
request.form.get("name")              # matches <input name="name">
return render_template("home.html", error=error, results=data)
return redirect(url_for("results", category=c, name=n))
{{ url_for('static', filename='style.css') }}
{% for row in results %} ... {% endfor %}
{% if results %} ... {% else %} no matches {% endif %}
```

### Route vs file
| URL | function | template |
|-----|----------|----------|
| `/` | `home` | `home.html` |
| `/results/<category>/<name>` | `results` | `results.html` |
| `/add` | `add_late` | `add.html` |

### POST not firing — checklist
- `methods=["GET", "POST"]` on that route
- `<form method="POST" action="...">` action matches the route
- submit is `<input type="submit">` inside the form
- `app.run()` still running; refresh after saving templates
